# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmeer-Siddiqui125/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-04 — Search Intelligence Data Contract

This notebook defines and verifies the data contract for the Search Intelligence classification lane.

The goal is to classify content into higher- and lower-engagement groups using information that is available at the prediction decision moment.

Development and verification use the March 2026 warehouse partition (`2026-03`). The final month is treated as a sealed outcome/test window.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

### Unit of analysis

One row represents the performance of **one content item for one client on one report date** in the daily performance warehouse.

The grain is therefore:

**`report_date × client_hash_id × content_hash_id`**

### Time window

I will use the **March 2026 panel (`2026-03`)** for development and verification.

March 2026 contains daily observations from **2026-03-01 through 2026-03-31**.

I use March rather than the final month because the final month is treated as a sealed outcome/test window.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
march_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").fetchone()

print("March row count:", march_check[0])
print("Date range:", march_check[1], "to", march_check[2])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March row count: 9841378
Date range: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The five candidate features for the engagement classification task are:

1. `gsc_impressions` — search visibility available before the decision moment.
2. `gsc_clicks` — search clicks available before the decision moment.
3. `gsc_avg_position` — average search position available before the decision moment.
4. `ga4_sessions` — sessions available before the decision moment when GA4 data is available.
5. `ga4_engaged_sessions` — engaged sessions available before the decision moment when GA4 data is available.

### Label / proxy

The prediction target is **future engagement level**.

The label will represent whether a content item has higher or lower engagement based on an engagement outcome measured after the feature observation period.

The label is not used as an input feature.

### Context

- `report_date` — identifies the observation date.
- `client_hash_id` — identifies the client and can be used for grouping or train/test splitting.
- `content_hash_id` — identifies the content item.
- `client_has_gsc` — indicates whether the client has GSC data.
- `client_has_ga4` — indicates whether the client has GA4 data.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `ga4_data_available` — indicates whether GA4 data is available for the observation.

### Excluded

The following are deliberately excluded from the feature set:

- Label-derived engagement fields — they contain or are derived from the outcome being predicted and would cause target leakage.
- Future-period performance — information that would not be available at the prediction decision moment.
- `client_hash_id` and `content_hash_id` as model inputs — they are identifiers, not meaningful predictive measurements.
- The final June 2026 sample — it is reserved as a sealed outcome/test window rather than being used during development.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The contract is verified using three small queries on the March 2026 panel.

1. Confirm the row count and date window.
2. Confirm the stated grain: `report_date × client_hash_id × content_hash_id`.
3. Confirm that performance data is only treated as available when the corresponding availability flag is `TRUE`.

The queries below are used to verify the contract rather than assuming the schema behaves as expected.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchall()

print("Duplicate grain rows:", grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows: []


In [4]:
availability_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").fetchone()

print("Total March rows:", availability_check[0])
print("Rows with GSC available:", availability_check[1])
print("Rows with GA4 available:", availability_check[2])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total March rows: 9841378
Rows with GSC available: 3611061
Rows with GA4 available: 413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.